<a href="https://colab.research.google.com/github/casper-justus/swahili-gpt/blob/main/inference_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇰🇪 Swahili GPT — Interactive Inference Demo

Generate Kiswahili text from a trained checkpoint, entirely in Colab.
No local setup required — just mount your Google Drive and run all cells.

---

## 📋 Before You Start

### ✅ Step 1: Enable the GPU
> Go to **Runtime → Change runtime type → T4 GPU → Save**.
> The model will still load on CPU but generation will be very slow.

### ✅ Step 2: Make sure your checkpoint exists on Drive
> Your checkpoint folder should be at:
> `/MyDrive/MiniGPT_Kiswahili_Checkpoints/`
> with a subfolder like `5000/` or `10000/` inside it.
> Also ensure `kenya_tokenizer.json` is in that same folder.

### ✅ Step 3: Run all cells top to bottom
> Then scroll to **Cell 5** to type your prompt and generate text interactively.


## Cell 1 — Mount Google Drive
Connects this notebook to your Google Drive so the checkpoint and tokenizer can be loaded.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints"

# Verify the checkpoint folder exists
if not os.path.exists(CKPT_DIR):
    raise FileNotFoundError(f"Checkpoint folder not found: {CKPT_DIR}\nMake sure you have trained the model first.")

print(f"✅ Drive mounted. Checkpoint dir: {CKPT_DIR}")
print(f"   Contents: {os.listdir(CKPT_DIR)}")


Mounted at /content/drive
✅ Drive mounted. Checkpoint dir: /content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints
   Contents: ['kenyan_tokenizer.json', '45000', '50000', '55000']


## Cell 2 — Install Dependencies
Installs the same pinned library versions used during training to guarantee compatibility.
> ⚠️ This takes 2–3 minutes. The long output is normal.


In [3]:
!pip uninstall -y jax jaxlib flax optax orbax-checkpoint -q
!pip install "jax[cuda12]" flax optax orbax-checkpoint tokenizers -q
print("✅ Dependencies installed.")


✅ Dependencies installed.


## Cell 3 — Load Model Architecture & Tokenizer
Rebuilds the exact same model architecture that was used during training, then loads the saved weights from your Drive checkpoint.

> ⚠️ The model config below **must match your training hyperparameters exactly**.
> If you changed `EMB_SIZE`, `NUM_HEADS`, or `NUM_LAYERS` during training, update them here too.


In [27]:
import jax
import jax.numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tokenizers import Tokenizer

# ── Model config (must match training) ─────────────────────────────
SEQ_LEN    = 1024
EMB_SIZE   = 512
NUM_HEADS  = 8
NUM_LAYERS = 6

# ── Architecture definition ──────────────────────────────────
class Block(nnx.Module):
    def __init__(self, emb_size, num_heads, rngs):
        self.ln_1 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(num_heads=num_heads, in_features=emb_size, decode=False, rngs=rngs)
        self.ln_2 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.mlp  = nnx.Sequential(
            nnx.Linear(emb_size, 4 * emb_size, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * emb_size, emb_size, rngs=rngs)
        )
    def __call__(self, x, mask):
        x = x + self.attn(self.ln_1(x), mask=mask)
        x = x + self.mlp(self.ln_2(x))
        return x

class MiniGPT(nnx.Module):
    def __init__(self, vocab_size, seq_len, emb_size, num_heads, num_layers, rngs):
        self.token_emb = nnx.Embed(vocab_size, emb_size, rngs=rngs)
        self.pos_emb   = nnx.Embed(seq_len, emb_size, rngs=rngs)
        self.blocks    = nnx.Sequential(*[Block(emb_size, num_heads, rngs) for _ in range(num_layers)])
        self.ln_f      = nnx.LayerNorm(emb_size, rngs=rngs)
        self.lm_head   = nnx.Linear(emb_size, vocab_size, rngs=rngs)
    def __call__(self, idx):
        b, t = idx.shape
        pos  = jnp.arange(0, t, dtype=jnp.int32)[None, :]
        x    = self.token_emb(idx) + self.pos_emb(pos)
        mask = nnx.make_causal_mask(jnp.ones((b, t)))
        for block in self.blocks.layers:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))

# ── Load tokenizer ──────────────────────────────────────────
print("Loading tokenizer...")
tokenizer  = Tokenizer.from_file(f"{CKPT_DIR}/kenyan_tokenizer.json")
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"✅ Tokenizer loaded. Vocab size: {VOCAB_SIZE}")

# ── Build model shell ──────────────────────────────────────────
print("Building model...")
rngs  = nnx.Rngs(0)
model = MiniGPT(VOCAB_SIZE, SEQ_LEN, EMB_SIZE, NUM_HEADS, NUM_LAYERS, rngs)

# ── Restore weights from checkpoint ───────────────────────────
# ── Restore weights from checkpoint ───────────────────────────
options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
mngr = ocp.CheckpointManager(CKPT_DIR, options=options)

start_step = 0

if mngr.latest_step() is not None:
    start_step = mngr.latest_step()
    print(f"Found existing checkpoint! Resuming from step {start_step}...")

    # Build a dummy optimizer to match the checkpoint structure
    import optax
    tx           = optax.adamw(learning_rate=3e-4)
    optimizer    = nnx.Optimizer(model, tx, wrt=nnx.Param)

    _, model_state = nnx.split(model)
    _, opt_state   = nnx.split(optimizer)
    state_tree     = {'model': model_state, 'opt': opt_state}

    # Pass the full state tree as target — fixes both topology mismatch
    # and the tree structure mismatch errors
    restored = mngr.restore(
        start_step,
        args=ocp.args.StandardRestore(state_tree)
    )

    nnx.update(model, restored['model'])
    # We don't update optimizer — we only needed it to match the tree shape
    print(f"✅ Model weights restored from step {start_step}.")
else:
    print("No checkpoint found. Starting fresh from Step 0.")

Loading tokenizer...
✅ Tokenizer loaded. Vocab size: 10000
Building model...
Found existing checkpoint! Resuming from step 55000...
✅ Model weights restored from step 55000.


## Cell 4 — Generation Function
Defines the `generate()` function using **top-k sampling with temperature scaling**.

**How the sampling parameters work:**
| Parameter | Effect | Recommended range |
|---|---|---|
| `temperature` | Controls randomness. Lower = repetitive but safe. Higher = creative but risky. | `0.7 – 1.0` |
| `top_k` | Only picks from the top K most likely next tokens. | `30 – 60` |
| `max_new_tokens` | How many new tokens to generate after the prompt. | `50 – 200` |

> ⚠️ **First generation call will take ~30–60 seconds** for JAX JIT compilation. All subsequent calls will be instant.


In [28]:
import jax
import jax.numpy as jnp

def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    """Generate Kiswahili text from a prompt using top-k sampling."""
    rng    = jax.random.PRNGKey(42)
    tokens = tokenizer.encode(prompt).ids

    for _ in range(max_new_tokens):
        ctx    = tokens[-SEQ_LEN:]
        logits = model(jnp.array([ctx]))[0, -1, :] / temperature
        top_logits, top_idx = jax.lax.top_k(logits, top_k)
        chosen  = jax.random.categorical(rng, top_logits)
        next_tok = int(top_idx[chosen])
        tokens.append(next_tok)
        rng = jax.random.fold_in(rng, next_tok)

    return tokenizer.decode(tokens)

print("✅ generate() function ready.")
print("   First call will take ~30-60s for JIT compilation. Subsequent calls are instant.")


✅ generate() function ready.
   First call will take ~30-60s for JIT compilation. Subsequent calls are instant.


## Cell 5 — ✨ Generate Text (Edit This Cell)
This is the cell you run repeatedly. Edit `PROMPT`, `TEMPERATURE`, and `MAX_TOKENS` to experiment.

**Prompt ideas to try:**
- `"Habari za asubuhi"` (Good morning news)
- `"Rais wa Kenya alisema"` (The President of Kenya said)
- `"Watoto wa shule"` (School children)
- `"Mvua ilikuwa inanyesha"` (It was raining)
- `"Siku moja, mtu mmoja"` (One day, a person)


In [40]:
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    enc        = tokenizer.encode(prompt)
    prompt_len = len(enc.ids)
    idx        = jnp.array([enc.ids], dtype=jnp.int32)
    key        = jax.random.PRNGKey(42)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -SEQ_LEN:]
        logits   = model(idx_cond)
        logits   = logits[0, -1, :] / temperature

        # Repetition penalty
        generated_ids = jnp.unique(idx[0])
        penalty_mask  = jnp.zeros(VOCAB_SIZE).at[generated_ids].set(1.0)
        logits        = jnp.where(penalty_mask > 0, logits / 1.3, logits)

        top_k_logits, top_k_indices = jax.lax.top_k(logits, top_k)
        probs       = jax.nn.softmax(top_k_logits)
        key, subkey = jax.random.split(key)
        sampled     = jax.random.choice(subkey, top_k_indices, p=probs)
        idx         = jnp.concatenate([idx, sampled[None, None]], axis=1)

    new_token_ids = idx[0, prompt_len:].tolist()

    # Decode the full sequence at once — the tokenizer knows where spaces go
    output = tokenizer.decode(new_token_ids)
    return output.strip()

# Run it
output = generate(model, tokenizer, PROMPT, MAX_TOKENS, TEMPERATURE, TOP_K)
print(output)

ya saa na nusu mwaka mmoja ( ambapo ma ru ) ni tukio la kawaida katika nchi nyingine . Kwa upande wa kaskazini , ma ru dio mbalimbali yame gawanyika kulingana na idadi kubwa sana , lakini pia wana tumia vi wanja vin ne nyuma kwa mchezo wake , basi hata hivyo , si tena kina cheza mechi nyingi tu na kushinda mashindano makubwa . K ulikuwa na timu nne pekee kabisa hivi karibuni . Mnamo [] US M ili tangaza kuwa taifa lenye haki zaidi kati ya watu milioni kumi wali zo pata kutoka kwenye orodha yao wenyewe tu


## Cell 6 — Batch Test Multiple Prompts
Run several prompts at once to quickly evaluate model quality across different topics.


In [41]:
test_prompts = [
    "Habari za asubuhi",
    "Leo hali ya hewa ni",
    "Serikali imetangaza kuwa",
    "Mvua ilikuwa inanyesha",
    "Siku moja, mtu mmoja",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print('='*60)
    output = generate(model, tokenizer, prompt, max_new_tokens=60, temperature=0.8, top_k=40)
    print(output)



PROMPT: Habari za asubuhi
ya saa na nusu mwaka mmoja ( ambapo ma ru ) ni tukio la kawaida katika nchi nyingine . Kwa upande wa kaskazini , ma ru dio mbalimbali yame gawanyika kulingana na idadi kubwa sana , lakini pia wana tumia vi wanja vin ne nyuma kwa mchezo wake , basi hata hivyo , si tena kina cheza mechi nyingi tu

PROMPT: Rais wa Kenya alisema
: " S ke mbo ni alikuwa mwana jeshi , mchezaji mpira na kocha . Ali pewa tuzo ya '' Young African s Award ''. * [] - K er ry Tar th ri no ( mshindi wa Tuzo za Nobel ) * 2017 – 17 Mei - G un ze b Jamii : Wachezaji mpira wa Norwei ''' Ki

PROMPT: Watoto wa shule
ya upili katika nchi kama vile Jimbo la Uchaguzi la T es cen , na pia , akiwa ni m hitimu mpa suaji mdogo zaidi kwa miaka mitatu . Jamii : Waliozaliwa 1999 Jamii : watu walio hai Jamii : wanasiasa mpira wa Uganda alt = Mwana soka | thumb |[[ Ki labu maalum cha mwisho ambacho kina onyeshwa kwenye

PROMPT: Mvua ilikuwa inanyesha
sana aina mbalimbali . * '' The co pti ve risti an life 